## Unstructured Data Cleaning

In this notebook we will apply different generic transformations on images, the only unstructured data we have. Specifically, we will remove corrupted images and normalize their size, while intentionally avoiding resolution changes to prevent information loss. All these transformations will be performed distributedly using Apache Spark.

### 1. Unified Dependencies & Environment Setup
To ensure pipeline maintainability and prevent session leakage, all third-party libraries (`boto3`, `PIL`), environment variables (`dotenv`), and core catalog schemas are consolidated at the entry point. 

In [1]:
import os
import io
import boto3
from PIL import Image
from dotenv import load_dotenv

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType

# Load environment variables from .env file
load_dotenv()

# Retrieve MinIO/S3 configuration credentials
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

# Initialize a Boto3 S3 client for MinIO interactions (used for metadata checks/validation)
s3 = boto3.client(
    "s3",
    endpoint_url=endpoint,
    aws_access_key_id=access_key,
    aws_secret_access_key=secret_key,
)

In [2]:
# Check that the number of images in both zones is the same
def count_images(bucket_name, prefix):
    count = 0

    paginator = s3.get_paginator("list_objects_v2")

    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        if "Contents" in page:
            for obj in page["Contents"]:
                key = obj["Key"]

                # skip folders / hidden/system files
                if not key.endswith("/") and not key.startswith(("_", ".")):
                    count += 1

    return count

landing_count = count_images("landing-zone", "persistent-landing/unstructured/image/")
trusted_count = count_images("trusted-zone", "unstructured/image/")

print("Landing zone images:", landing_count)
print("Trusted zone images:", trusted_count)

Landing zone images: 6862
Trusted zone images: 6674


### 2. Optimized SparkSession Configuration for MinIO/S3A
We configure the Apache Spark backend specifically for high-throughput object store interactions via the S3A connector. 

Key architectural parameters configured below include:
* **Delta Lake Catalog Mapping:** Enabling ACID transaction support over unstructured object registries.
* **Explicit AWS Credentials Provider:** Standardizing on `SimpleAWSCredentialsProvider` to minimize metadata service timeout loops on distributed worker tasks.
* **Hadoop Configuration Sanitization:** A critical runtime override loop that strips duration suffixes (e.g., converting "60s" or "1h" to pure numeric format), resolving internal timestamp parsing bugs common in specific Spark-Hadoop client combinations.

In [3]:
# Define Delta Lake version for compatibility
DELTA_VERSION = "4.1.0"

# Initialize SparkSession with Delta Lake and S3A (MinIO) configurations
spark = SparkSession.builder \
    .appName("trusted_zone_unstructured_processing") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", f"org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,io.delta:delta-spark_2.13:{DELTA_VERSION}") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "60000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "60000") \
    .getOrCreate()

# Normalize Hadoop configuration values (e.g., converting "60s" to "60") to prevent version mismatch errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key = item.getKey()
    value = item.getValue()

    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        numeric_value = "".join(char for char in value if char.isdigit())
        hadoop_conf.set(key, numeric_value)

### 3. Catalog Deserialization & Data Deduplication
Instead of executing an expensive distributed file system scan (`dbutils.fs.ls` or `s3_client.list_objects`), we query the decoupled Metadata Catalog Delta table from the `landing-zone`.

We optimize downstream computation by applying early filtering predicates:
1.  **Type Enforcement:** Dropping non-image assets immediately.
2.  **Schema Projection:** Parsing the raw metadata JSON blob into structured columns to prevent repeated string parsing.
3.  **Idempotence & Quality Control:** Excluding corrupted flags and executing a global deduplication based on the **MD5 hash checksum** to ensure each unique binary is processed exactly once.

In [4]:
# Pack the extracted variables into a dictionary and broadcast them across the cluster.
# This avoids Python closure serialization bottlenecks (PicklingError) in distributed environments.
minio_creds = {
    "endpoint": endpoint,
    "access_key": access_key,
    "secret_key": secret_key
}
credentials_broadcast = spark.sparkContext.broadcast(minio_creds)
# Define the schema for the metadata JSON blob
blob_schema = StructType([
    StructField("label", StringType(), True),
    StructField("url", StringType(), True),
    StructField("file_size_bytes", IntegerType(), True),
    StructField("content_type", StringType(), True),
    StructField("width", IntegerType(), True),
    StructField("height", IntegerType(), True),
    StructField("aspect_ratio", DoubleType(), True),
    StructField("image_mode", StringType(), True),
    StructField("is_corrupted", BooleanType(), True),
    StructField("md5", StringType(), True)
])

# Load the file catalog Delta table from the landing zone
catalog_path = "s3a://landing-zone/persistent-landing/structured/file_catalog/"
catalog_df = spark.read.format("delta").load(catalog_path)

# Filter for images, parse metadata, and discard corrupted or duplicated files
cleaned_image_df = catalog_df.filter(F.col("file_type") == "Image") \
    .withColumn("meta", F.from_json(F.col("metadata_blob"), blob_schema)) \
    .filter(F.col("file_path").isNotNull()) \
    .filter(F.col("meta.is_corrupted") == False) \
    .withColumn("md5", F.col("meta.md5")) \
    .dropDuplicates(["md5"])

# Select and project essential columns required for the image processing pipeline
paths_df = cleaned_image_df.select(
    F.expr("concat('s3a://landing-zone/', file_path)").alias("full_raw_path"),
    F.col("file_id").alias("id"),                 
    F.col("file_path").alias("file_path"),       #physical path specifically for S3 key lookups
    F.col("meta.image_mode").alias("image_mode"),
    F.col("meta.width").alias("width"),
    F.col("meta.height").alias("height")
)

# Convert to RDD for high-performance processing
paths_rdd = paths_df.repartition(8).rdd

# Repartition the DataFrame to optimize parallel processing across Spark workers
# Note: Adjust the partition size (8) based on your cluster's CPU core count
paths_rdd = paths_df.repartition(8).rdd

### 4. Distributed Image Processing Strategy: Fast-Path vs. Compute-Path
Processing large-scale image corpuses using standard PySpark UDFs (User-Defined Functions) introduces severe JVM-to-Python serialization bottlenecks. To maximize cluster utilization, we leverage `mapPartitions()`.

#### Performance Architecture:
* **Connection Pooling:** The `boto3` client is instantiated *inside* the executor partition loop rather than driver-side, preventing distributed connection serialization errors (`PicklingError`).
* **The Fast-Path (Zero-Compute):** If an incoming image already matches the Target Specification (512x512, RGB mode, PNG format), the worker issues an out-of-band `copy_object` command directly to MinIO. This executes as a metadata-only copy, bypassing network IO and Pillow memory footprint entirely.
* **The Compute-Path:** Non-standard images are streamed into memory via a byte buffer, standardized using Pillow, and written back to the `trusted-zone` asynchronously.

In [5]:
def transform_and_upload_image(rows):
    """
    Spark MapPartitions function to read, transform, and upload images to the trusted zone.
    Decouples business logical 'id' from the physical S3 keys for robust asset tracking.
    """
    import os
    import io
    import boto3
    from PIL import Image

    # Retrieve infrastructure credentials from the broadcast block safely
    creds = credentials_broadcast.value
    
    partition_s3_client = boto3.client(
        "s3",
        endpoint_url=creds["endpoint"],
        aws_access_key_id=creds["access_key"],
        aws_secret_access_key=creds["secret_key"]
    )
    
    dest_bucket = "trusted-zone"
    
    for row in rows:
        try:
            image_id = row["id"]           
            src_key = row["file_path"]         
            orig_mode = row["image_mode"]  
            orig_w = row["width"]         
            orig_h = row["height"]         
            
            orig_ext = src_key.rsplit(".", 1)[-1].lower()
            base_key = src_key.rsplit(".", 1)[0] + ".png"
            dest_key = base_key.replace("persistent-landing/", "") 
            
            # Fast Path: Zero-compute S3 copy for already standardized images
            if orig_w == 512 and orig_h == 512 and orig_mode == "RGB" and orig_ext == "png":
                partition_s3_client.copy_object(
                    Bucket=dest_bucket,
                    Key=dest_key,
                    CopySource={"Bucket": "landing-zone", "Key": src_key},
                    ContentType="image/png"
                )
                # 🚀 Yield the true logical ID back to the driver
                yield {"id": image_id, "trusted_path": f"s3a://{dest_bucket}/{dest_key}", "status": "DIRECT_COPY"}
                continue

            # Compute Path: Download, transform via Pillow in-memory, and upload
            obj = partition_s3_client.get_object(Bucket="landing-zone", Key=src_key)
            image_data = obj["Body"].read()

            img = Image.open(io.BytesIO(image_data))

            if orig_mode != "RGB":
                img = img.convert("RGB")

            img = img.resize((512, 512))

            buffer = io.BytesIO()
            img.save(buffer, format="PNG")
            buffer.seek(0)

            partition_s3_client.put_object(
                Bucket=dest_bucket,
                Key=dest_key,
                Body=buffer.getvalue(),
                ContentType="image/png"
            )
            
            #Yield the true logical ID back to the driver
            yield {"id": image_id, "trusted_path": f"s3a://{dest_bucket}/{dest_key}", "status": "PILLOW_TRANSFORMED"}

        except Exception as e:
            # Yield resilience logging context tied to the true ID
            yield {"id": row["id"], "trusted_path": None, "status": f"FAILED: {str(e)}"}

In [6]:
# Action: Trigger the Spark execution DAG to process and load images
result_rdd = paths_rdd.mapPartitions(transform_and_upload_image)
processing_results = result_rdd.collect()

# Display a subset of the processing logs for verification
print(f"Pipeline executed. Processed {len(processing_results)} images. Sample logs:")
for log in processing_results[:5]:
    print(log)

Pipeline executed. Processed 6674 images. Sample logs:
{'id': 'image_1778924713160.jpg', 'trusted_path': 's3a://trusted-zone/unstructured/image/image_1778924713160.png', 'status': 'PILLOW_TRANSFORMED'}
{'id': 'image_1778924608224.jpg', 'trusted_path': 's3a://trusted-zone/unstructured/image/image_1778924608224.png', 'status': 'PILLOW_TRANSFORMED'}
{'id': 'image_1778924637417.jpg', 'trusted_path': 's3a://trusted-zone/unstructured/image/image_1778924637417.png', 'status': 'PILLOW_TRANSFORMED'}
{'id': 'image_1778924636206.jpg', 'trusted_path': 's3a://trusted-zone/unstructured/image/image_1778924636206.png', 'status': 'PILLOW_TRANSFORMED'}
{'id': 'image_1778924608807.jpg', 'trusted_path': 's3a://trusted-zone/unstructured/image/image_1778924608807.png', 'status': 'PILLOW_TRANSFORMED'}


### 5. Post-Load Data Integrity & Validation Loop
To close the data pipeline lifecycle, we run an end-to-end audit. Using a paginated S3 client loop, we verify the absolute count of processed physical objects sitting in the `trusted-zone`. 

This independent validation decouples from Spark’s internal lineage tracking to guarantee that what was processed in-memory matches the physical state of the Object Storage sink.

In [7]:
# Validation step: Count processed files in the Trusted Zone
def count_s3_objects(bucket_name, prefix):
    """Paginates through an S3 bucket prefix and returns the count of valid files."""
    count = 0
    paginator = s3.get_paginator("list_objects_v2")
    
    for page in paginator.paginate(Bucket=bucket_name, Prefix=prefix):
        for obj in page.get("Contents", []):
            key = obj["Key"]
            # Exclude directory placeholders and hidden files
            if not key.endswith("/") and not key.startswith(("_", ".")):
                count += 1
    return count

trusted_images_count = count_s3_objects("trusted-zone", "unstructured/image/")
print(f"[Validation] Total standardized images in Trusted Zone: {trusted_images_count}")

[Validation] Total standardized images in Trusted Zone: 6674


### 6. Dynamic Meta-Catalog Materialization
A mature unstructured data pipeline must not only move bytes but also enrich the state of the data catalog. 

Instead of treating the processing logs as a simple tracking table, this stage executes an **inner join** with the upstream metadata schema to build an enriched **Image Asset Catalog**. 

#### Data Governance Strategy:
1. **Preserving Lineage:** Core business dimensions like `label` and `source_url` are retained and linked directly to the new physical storage paths (`trusted_path`).
2. **Feature Stores Readiness:** By declaring explicit physical dimensions (`current_width`, `current_height`, `current_image_mode`), downstream Deep Learning feature pipelines (e.g., PyTorch DataLoaders) can query this table via Spark SQL to perform filtered server-side batching, eliminating client-side image validation overheads.

In [8]:
# Define the exact metrics schema reflecting the pure logical ID
result_schema = StructType([
    StructField("id", StringType(), True),           # 🚀 Now perfectly matches 'file_id' format
    StructField("trusted_path", StringType(), True), # Unified target path in trusted-zone
    StructField("status", StringType(), True)        # Operational execution status
])

# Convert results into a Spark DataFrame and filter out any failed partitions
execution_results_df = spark.createDataFrame(processing_results, schema=result_schema) \
    .filter(F.col("status").isin("DIRECT_COPY", "PILLOW_TRANSFORMED"))

# [Relational Enrichment Table Join]:
# Join on the clean 'id' string to recover historical metadata fields seamlessly
trusted_catalog_df = execution_results_df.join(
    cleaned_image_df.select(
        F.col("file_id").alias("id"),         # 🚀 Connect using the pure image ID
        F.col("file_path").alias("raw_source_path"), # Keep raw path as an audit dimension
        F.col("meta.label").alias("label"),   # Recover the business label
        F.col("meta.url").alias("source_url") # Recover the lineage source URL
    ),
    on="id",
    how="inner"
)

# Append target standardized physical asset specifications
trusted_catalog_df = trusted_catalog_df \
    .withColumn("current_width", F.lit(512)) \
    .withColumn("current_height", F.lit(512)) \
    .withColumn("current_image_mode", F.lit("RGB")) \
    .withColumn("current_format", F.lit("PNG")) \
    .withColumn("processed_at", F.current_timestamp())

# Display the pristine Data Asset Catalog
# As requested, 'id' is now the pure system ID, not the full physical path string!
trusted_catalog_df.select(
    "id","raw_source_path" ,"label", "source_url", "current_width", "current_image_mode", "trusted_path"
).show(5, truncate=False)

+-----------------------+-------------------------------------------------------------+-----+-------------------------------------------------------------+-------------+------------------+-------------------------------------------------------------+
|id                     |raw_source_path                                              |label|source_url                                                   |current_width|current_image_mode|trusted_path                                                 |
+-----------------------+-------------------------------------------------------------+-----+-------------------------------------------------------------+-------------+------------------+-------------------------------------------------------------+
|image_1778924713160.jpg|persistent-landing/unstructured/image/image_1778924713160.jpg|dew  |https://www.kaggle.com/datasets/jehanbhathena/weather-dataset|512          |RGB               |s3a://trusted-zone/unstructured/image/image_1778924713160.p

In [9]:
# Declare the target physical path for the Trusted Zone Asset Catalog
trusted_catalog_s3_path = "s3a://trusted-zone/file_catalog/"

# Commit the catalog to object storage using Delta format to enforce ACID transaction logs,
# enabling 'overwrite' for pipeline re-runs while safely allowing schema evolution if needed
trusted_catalog_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .save(trusted_catalog_s3_path)

print(f"[Success] Enriched Enterprise Data Catalog written to: {trusted_catalog_s3_path}")

[Success] Enriched Enterprise Data Catalog written to: s3a://trusted-zone/file_catalog/
